# WC 2026 Hydration Breaks — full reproducible analysis
Companion to *Do In-Match Hydration Breaks Alter Match Momentum?* Run from the project root.

Two candidate designs for the break window: **clock-aligned** (`analysis_gap.py`) and **play-aligned / frozen**
(`analysis_designB.py`). Within-match case-crossover is primary; external controls (`external_dom.py`) are a
robustness check (tight-caliper matching only) with a positivity limitation. xG is the alignment-invariant
adjudicator.

## 1 · Within-match clock-aligned builder + ladder

In [1]:
"""
Consolidated, break-gap-corrected within-match case-crossover pipeline.
Key correction (reviewer comment 1): the SofaScore Attack Momentum series is indexed by nominal
match minute with the clock running through the ~3-min hydration break, so in-break minutes carry
spurious (decay-artefact) values. We therefore EXCLUDE a uniform 3-minute in-play window at each
break start c (minutes c, c+1, c+2) and measure the post-break outcome only after play resumes
(from c+3). The same 3-minute pseudo-break exclusion is applied to every control anchor, so the
treated/control contrast preserves an identical temporal structure.
Outputs: paper/figures/*.png, paper/results_ladder.txt
"""
import json, numpy as np, pandas as pd, statsmodels.formula.api as smf, warnings
warnings.filterwarnings('ignore')
exec(open('model_within.py').read().split('def fit(')[0])  # raw,goals,breaks,WB,mom_at,wm,wsl,sb,period,ACCITY,citym

GAP = 3                 # uniform in-play exclusion (minutes c, c+1, c+2); post-window starts at c+GAP
DROP = '15186769'       # France vs Iraq: only one break recorded (no H2 break) -> drop for 2-per-match balance

def valid_len(mid):
    """Momentum series must be consistent with nominal-minute indexing: ~90 (+trailing) for regulation,
    ~120 for extra time. Series far from these (e.g. 132, 136) carry resampling/padding artefacts that
    misalign the minute axis and the break location, so those matches are excluded."""
    if mid not in raw: return False
    L=len(raw[mid][1])
    return (80<=L<=100) or (118<=L<=128)

# oriented Elo of the dominant side
tr0=pd.DataFrame(json.load(open('data/treated_covariates.json')))
ELO=tr0.groupby('match_id')[['elo_home','elo_away']].first()
elo_h={str(k):v for k,v in ELO['elo_home'].items()}; elo_a={str(k):v for k,v in ELO['elo_away'].items()}

def gap_series(mid,c,H):
    """oriented momentum: pre-level/slope over [c-5,c-1]; post over [c+GAP, c+GAP+H-1]; None if invalid."""
    pre=wm(mid,c-5,c-1); sl=wsl(mid,c-5,c-1)
    if pre is None or sl is None: return None
    s,mo=raw[mid]; last=s+len(mo)-1
    lo, hi = c-5, c+GAP+H-1
    if lo<s or hi>last: return None
    if period(lo)!=period(hi): return None            # no half-time / period crossing
    o = 1 if pre>=0 else -1
    post=[mom_at(mid,c+GAP+k) for k in range(H)]; post=[v for v in post if v is not None]
    if not post: return None
    hs,a=sb(mid,c); dm=o*(hs-a)
    w=WB(mid); m=str(mid)
    if w is None or m not in elo_h: return None
    return dict(o=o, pre_level=o*pre, pre_slope=o*sl, margin=dm, wbgt=w,
                elo_gap=o*(elo_h[m]-elo_a[m]), Y=o*float(np.mean(post)))

def build(H):
    R=[]
    for mid,bks in breaks.items():
        if not valid_len(mid): continue                # drop corrupted-length series
        for c in bks:                                  # treated: break START minutes
            r=gap_series(mid,c,H)
            if r: R.append({'match_id':str(mid),'brk':1,'minute':c,'half':int(c>=45),**r})
    for mid,(s,mo) in raw.items():
        if not valid_len(mid): continue
        for c in range(s+5, s+len(mo)-1):              # control: candidate anchors
            if any(b-2<=c<=b+GAP+H for b in breaks.get(mid,[])): continue   # keep clear of real breaks+window
            if 43<=c<=48: continue                     # avoid half-time seam
            r=gap_series(mid,c,H)
            if r: R.append({'match_id':str(mid),'brk':0,'minute':c,'half':int(c>=45),**r})
    d=pd.DataFrame(R)
    d=d[d.match_id!=DROP].copy()
    d['m2']=d.minute**2; d['wbgt_c']=d.wbgt-d.wbgt.mean(); d['elo_c']=(d.elo_gap-d.elo_gap.mean())/100.0
    d['brk_margin']=d.brk*d.margin; d['brk_wbgt']=d.brk*d.wbgt_c; d['brk_elo']=d.brk*d.elo_c
    return d

if __name__=='__main__':
    d=build(10)
    nb=int(d.brk.sum()); nc=int((1-d.brk).sum()); nm=d.match_id.nunique()
    print(f"[gap-corrected] outcome=oriented mean momentum over [c+{GAP}, c+{GAP+9}] | break n={nb}, control n={nc}, matches={nm}")
    def ols(f): return smf.ols(f,data=d).fit(cov_type='cluster',cov_kwds={'groups':d.match_id})
    base="Y ~ C(match_id) + minute + m2 + C(half) + pre_level + pre_slope + margin"
    def L(r,t):
        p=r.params[t]; s=r.bse[t]; return f"{p:+.2f} [{p-1.96*s:+.2f},{p+1.96*s:+.2f}] (p={r.pvalues[t]:.2f})"
    M0=ols(base+" + brk"); M1=ols(base+" + brk + brk_wbgt"); M2=ols(base+" + brk + brk_margin")
    M3=ols(base+" + brk + brk_elo"); M4=ols(base+" + brk + brk_margin + brk_elo + brk_wbgt")
    print("M0 base     ", L(M0,'brk'))
    print("M1 +heat    ", L(M1,'brk'),"| xWBGT",L(M1,'brk_wbgt'))
    print("M2 +lead    ", L(M2,'brk'),"| xlead",L(M2,'brk_margin'))
    print("M3 +elo     ", L(M3,'brk'),"| xElo",L(M3,'brk_elo'))
    print("M4 full     ", L(M4,'brk'),"| xlead",L(M4,'brk_margin'),"| xElo",L(M4,'brk_elo'))
    def mixed(rhs): return smf.mixedlm("Y ~ "+rhs,data=d,groups=d["match_id"]).fit(method='lbfgs')
    rb="minute + m2 + C(half) + pre_level + pre_slope + margin"
    X1=mixed(rb+" + brk"); X2=mixed(rb+" + brk + brk_margin"); X3=mixed(rb+" + brk + brk_margin + brk_elo")
    print("X1 mixed    ", L(X1,'brk'))
    print("X2 mixed+lead", L(X2,'brk'),"| xlead",L(X2,'brk_margin'))
    print("X3 mixed+lead+elo", L(X3,'brk'),"| xlead",L(X3,'brk_margin'),"| xElo",L(X3,'brk_elo'))
    with open('paper/results_ladder.txt','w') as f:
        f.write(f"gap-corrected outcome = oriented mean momentum over post-resumption window [c+{GAP}, c+{GAP+9}]\n")
        f.write(f"break events n={nb}; control anchors n={nc}; matches={nm}; uniform {GAP}-min break exclusion\n\n")
        f.write("FIXED EFFECTS (within-match, cluster-robust):\n")
        for lab,r,extra in [("M0 base",M0,[]),("M1 +heat",M1,[('brk_wbgt','xWBGT/C')]),
                            ("M2 +lead",M2,[('brk_margin','xlead')]),("M3 +elo",M3,[('brk_elo','xElo/100')]),
                            ("M4 full",M4,[('brk_margin','xlead'),('brk_elo','xElo/100'),('brk_wbgt','xWBGT/C')])]:
            f.write(f"  {lab:10} break {L(r,'brk')}"+"".join(f" ; {nm2} {L(r,c)}" for c,nm2 in extra)+"\n")
        f.write("\nMIXED EFFECTS (random match intercept):\n")
        for lab,r,extra in [("X1 main",X1,[]),("X2 +lead",X2,[('brk_margin','xlead')]),
                            ("X3 +lead+elo",X3,[('brk_margin','xlead'),('brk_elo','xElo/100')])]:
            f.write(f"  {lab:12} break {L(r,'brk')}"+"".join(f" ; {nm2} {L(r,c)}" for c,nm2 in extra)+"\n")
    print("\nwrote paper/results_ladder.txt")


[gap-corrected] outcome=oriented mean momentum over [c+3, c+12] | break n=198, control n=3139, matches=99


M0 base      +0.26 [-2.50,+3.02] (p=0.85)
M1 +heat     +0.25 [-2.50,+3.00] (p=0.86) | xWBGT -0.23 [-0.83,+0.36] (p=0.44)
M2 +lead     +0.38 [-2.54,+3.31] (p=0.80) | xlead -0.90 [-3.63,+1.83] (p=0.52)
M3 +elo      +0.12 [-2.70,+2.93] (p=0.94) | xElo +1.09 [-0.45,+2.63] (p=0.17)
M4 full      +0.35 [-2.57,+3.26] (p=0.81) | xlead -2.19 [-4.88,+0.50] (p=0.11) | xElo +1.58 [+0.01,+3.16] (p=0.05)


X1 mixed     +0.26 [-2.45,+2.97] (p=0.85)
X2 mixed+lead +0.38 [-2.35,+3.11] (p=0.78) | xlead -0.90 [-2.99,+1.20] (p=0.40)
X3 mixed+lead+elo +0.36 [-2.37,+3.09] (p=0.80) | xlead -2.23 [-4.55,+0.09] (p=0.06) | xElo +1.59 [+0.39,+2.79] (p=0.01)

wrote paper/results_ladder.txt


## 2 · Figures + referent table

In [2]:
"""Gap-corrected figures + referent-sensitivity table + Table 1 descriptives."""
import json, numpy as np, pandas as pd, statsmodels.formula.api as smf, warnings
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
exec(open('analysis_gap.py').read().split("if __name__")[0])   # loaders + build + GAP + gap_series + elo
plt.rcParams.update({'font.family':'serif','font.size':11,'axes.titlesize':12,'axes.labelsize':11,
    'axes.spines.top':False,'axes.spines.right':False,'figure.dpi':140,'axes.grid':True,
    'grid.alpha':0.25,'grid.linewidth':0.6,'legend.frameon':False})
BLUE='#2f6db5'; RED='#c0392b'; INK='#1f2430'; MUT='#6b7280'; GOLD='#c98a1a'; GREEN='#2e7d5b'; FIG='paper/figures'

# ============================ FIG 1: sign-adjusted outcome WITH break gap ============================
mid=12813017; s,mo=raw[mid]; bk=[b for b in breaks[mid] if b<45][0]
mins=np.array(range(s,s+len(mo)),dtype=float); yraw=np.array(mo,dtype=float)
o=1 if wm(mid,bk-5,bk-1)>=0 else -1
# blank the in-play break window [bk, bk+GAP-1] so the curve shows a gap
def blанk(arr):
    a=arr.copy()
    for k in range(GAP):
        idx=np.where(mins==bk+k)[0]
        if len(idx): a[idx]=np.nan
    return a
yraw_g=blанk(yraw); yor=o*yraw; yor_g=blанk(yor)
fig,ax=plt.subplots(1,2,figsize=(10.4,3.9))
a=ax[0]; a.axhline(0,color=INK,lw=.8)
a.fill_between(mins,yraw_g,0,where=yraw_g>=0,color=BLUE,alpha=.35,linewidth=0)
a.fill_between(mins,yraw_g,0,where=yraw_g<0,color=RED,alpha=.35,linewidth=0)
a.plot(mins,yraw_g,color=INK,lw=1.0)
a.axvspan(bk-5,bk-1,color=GOLD,alpha=.20); a.axvspan(bk,bk+GAP-1,color='0.5',alpha=.35,hatch='//',ec='none')
a.set_xlim(5,60); a.set_title('(a) Raw signed attack momentum'); a.set_xlabel('Match minute'); a.set_ylabel('Momentum (home + / away $-$)')
a.text(bk-3,a.get_ylim()[1]*.92,'pre-break\nwindow',ha='center',va='top',fontsize=8.3,color=GOLD)
a.text(bk+0.6,a.get_ylim()[0]*.9,'break\n(no play)',ha='center',va='bottom',fontsize=8.0,color='0.3')
a.text(12,a.get_ylim()[1]*.7,'HOME',color=BLUE,fontsize=9,weight='bold'); a.text(12,a.get_ylim()[0]*.7,'AWAY',color=RED,fontsize=9,weight='bold')
b=ax[1]; b.axhline(0,color=INK,lw=.8)
b.fill_between(mins,yor_g,0,where=yor_g>=0,color=GREEN,alpha=.35,linewidth=0)
b.fill_between(mins,yor_g,0,where=yor_g<0,color=MUT,alpha=.25,linewidth=0)
b.plot(mins,yor_g,color=INK,lw=1.0)
b.axvspan(bk-5,bk-1,color=GOLD,alpha=.20); b.axvspan(bk,bk+GAP-1,color='0.5',alpha=.35,hatch='//',ec='none')
preL=o*wm(mid,bk-5,bk-1); postL=o*float(np.mean([mom_at(mid,bk+GAP+k) for k in range(10)]))
b.plot([bk-3],[preL],'o',color=GREEN,ms=5); b.plot([bk+GAP+4.5],[postL],'o',color=GREEN,ms=5)
b.annotate('',xy=(bk+GAP+4.5,postL),xytext=(bk-3,preL),arrowprops=dict(arrowstyle='->',color=INK,lw=1.1))
b.text(bk+GAP+0.5,(preL+postL)/2+5,'post-resumption\nmomentum',fontsize=8.0,color=INK)
b.set_xlim(5,60); b.set_title('(b) Sign-adjusted (dominant-side) momentum $Y_t=o\\cdot M_t$')
b.set_xlabel('Match minute'); b.set_ylabel('Momentum toward pre-break dominant side')
b.text(12,b.get_ylim()[1]*.7,'DOMINANT side (+)',color=GREEN,fontsize=8.5,weight='bold')
fig.tight_layout(); fig.savefig(f'{FIG}/fig1_orientation.png',bbox_inches='tight'); plt.close(fig)
print('fig1 (gap) done; o=%d preL=%.1f postL=%.1f'%(o,preL,postL))

# ============================ FIG 2: regression-to-mean WITH break gap ============================
def oriented(mid,c,lo=-5,hi=15):
    pre=wm(mid,c-5,c-1)
    if pre is None: return None
    o=1 if pre>=0 else -1; s,mo=raw[mid]; last=s+len(mo)-1
    if c+lo<s or c+hi>last or period(c+lo)!=period(c+hi): return None
    out=[]
    for k in range(lo,hi+1):
        if 0<=k<GAP: out.append(np.nan); continue         # blank in-break minutes
        v=mom_at(mid,c+k); out.append(o*v if v is not None else np.nan)
    return np.array(out)
K=np.arange(-5,16); Br=[]; Ct=[]
for mid,bks in breaks.items():
    if str(mid)==DROP or not valid_len(mid): continue
    for c in bks:
        r=oriented(mid,c); Br.append(r) if r is not None else None
for mid,(s,mo) in raw.items():
    if str(mid)==DROP or not valid_len(mid): continue
    for c in range(s+5,s+len(mo)-15):
        if any(b-2<=c<=b+GAP+15 for b in breaks.get(mid,[])): continue
        if 43<=c<=48: continue
        r=oriented(mid,c); Ct.append(r) if r is not None else None
Br=np.vstack(Br); Ct=np.vstack(Ct)
def mci(M): m=np.nanmean(M,0); se=np.nanstd(M,0)/np.sqrt(np.sum(np.isfinite(M),0)); return m,m-1.96*se,m+1.96*se
mb,lb,hb=mci(Br); mc,lc,hc=mci(Ct)
fig,ax=plt.subplots(figsize=(7.2,4.4))
ax.axvspan(-0.4,GAP-0.6,color='0.5',alpha=.30,hatch='//',ec='none')
ax.axhline(0,color=INK,lw=.7)
ax.plot(K,mb,color=RED,lw=1.8,label=f'break minutes (n={Br.shape[0]})',marker='o',ms=3)
ax.fill_between(K,lb,hb,color=RED,alpha=.15)
ax.plot(K,mc,color=BLUE,lw=1.8,label=f'non-break control anchors (n={Ct.shape[0]})',marker='s',ms=3)
ax.fill_between(K,lc,hc,color=BLUE,alpha=.12)
ax.set_xlabel('Minutes relative to break start (in-break minutes blanked)'); ax.set_ylabel('Sign-adjusted momentum (dominant side)')
ax.set_title('The fade is mean reversion: treated and control decay alike')
ax.text((GAP-1)/2-0.2,ax.get_ylim()[1]*.96,'break\n(no play)',fontsize=8,va='top',ha='center',color='0.3')
ax.legend(loc='upper right')
fig.tight_layout(); fig.savefig(f'{FIG}/fig2_regression_to_mean.png',bbox_inches='tight'); plt.close(fig)
print('fig2 (gap) done; pre(-5..-1) brk=%.1f ctrl=%.1f | post(+3..+7) brk=%.1f ctrl=%.1f'%(
      np.nanmean(mb[0:5]),np.nanmean(mc[0:5]),np.nanmean(mb[8:13]),np.nanmean(mc[8:13])))

# ============================ FIG 3 (was fig4): break-minute windows ============================
bm=[b for mid,bks in breaks.items() if str(mid)!=DROP for b in bks]
fig,ax=plt.subplots(figsize=(7.2,3.4))
ax.hist([b for b in bm if b<45],bins=range(18,32),color=GREEN,alpha=.75,label='first-half breaks')
ax.hist([b for b in bm if b>=45],bins=range(62,78),color=GOLD,alpha=.75,label='second-half breaks')
ax.set_xlabel('Break start minute'); ax.set_ylabel('Number of break events')
ax.set_title('Breaks occur only in two narrow windows ($\\approx$23$^\\prime$ and $\\approx$68$^\\prime$)')
ax.legend()
fig.tight_layout(); fig.savefig(f'{FIG}/fig3_break_minutes.png',bbox_inches='tight'); plt.close(fig)
print('fig3 (break minutes) done')

# ============================ Referent / time-trend sensitivity table ============================
d=build(10); d['brk_margin']=d.brk*d.margin
def fit(dd,g):
    m=smf.ols(f"Y ~ {g} + C(half) + pre_level + pre_slope + margin + brk + brk_margin",
              data=dd).fit(cov_type='cluster',cov_kwds={'groups':dd.match_id})
    ci=lambda t:(m.params[t],m.params[t]-1.96*m.bse[t],m.params[t]+1.96*m.bse[t])
    return ci('brk'),ci('brk_margin'),int((1-dd.brk).sum())
rows=[]
for lab,g in [("linear","minute"),("quadratic","minute + m2"),("cubic","minute + m2 + I(minute**3)"),("per-minute dummies","C(minute)")]:
    b,gm,ncc=fit(d,g); rows.append(("A",lab,b,gm,ncc))
def band(dd,a,bb,cc,e):
    keep=((dd.brk==0)&(((dd.half==0)&dd.minute.between(a,bb))|((dd.half==1)&dd.minute.between(cc,e))))
    return dd[(dd.brk==1)|keep].copy()
for lab,(a,bb,cc,e) in [("adjacent 15-32 | 60-77",(15,32,60,77)),("mid 11-40 | 56-85",(11,40,56,85)),
                        ("far 6-15 | 73-78",(6,15,73,78)),("all eligible",(0,44,45,120))]:
    dd=band(d,a,bb,cc,e); b,gm,ncc=fit(dd,"minute + m2"); rows.append(("B",lab,b,gm,ncc))
with open('paper/referent_table.txt','w') as f:
    for grp,lab,b,gm,ncc in rows:
        f.write(f"{grp} {lab:26} beta {b[0]:+.2f}[{b[1]:+.2f},{b[2]:+.2f}]  gamma {gm[0]:+.2f}[{gm[1]:+.2f},{gm[2]:+.2f}]  nctrl={ncc}\n")
print('referent table written'); print(open('paper/referent_table.txt').read())


fig1 (gap) done; o=-1 preL=41.4 postL=7.1


fig2 (gap) done; pre(-5..-1) brk=20.4 ctrl=19.5 | post(+3..+7) brk=6.5 ctrl=8.1


fig3 (break minutes) done


referent table written
A linear                     beta +0.26[-2.69,+3.21]  gamma -0.17[-2.77,+2.43]  nctrl=3139
A quadratic                  beta +0.50[-2.42,+3.42]  gamma -0.15[-2.76,+2.46]  nctrl=3139
A cubic                      beta +0.92[-2.37,+4.21]  gamma -0.18[-2.79,+2.44]  nctrl=3139
A per-minute dummies         beta -1.70[-9.01,+5.61]  gamma -0.12[-2.65,+2.41]  nctrl=3139
B adjacent 15-32 | 60-77     beta -0.31[-4.52,+3.90]  gamma +0.55[-1.98,+3.08]  nctrl=1342
B mid 11-40 | 56-85          beta +1.07[-2.68,+4.82]  gamma +0.48[-2.35,+3.30]  nctrl=2134
B far 6-15 | 73-78           beta -5.14[-13.46,+3.19]  gamma -2.34[-8.78,+4.10]  nctrl=990
B all eligible               beta +0.50[-2.42,+3.42]  gamma -0.15[-2.76,+2.46]  nctrl=3139



## 3 · Clock-aligned coefficient table (Table 3)

In [3]:
import numpy as np, statsmodels.formula.api as smf, warnings
warnings.filterwarnings('ignore')
exec(open('analysis_gap.py').read().split('if __name__')[0])
d=build(10)
base='C(match_id) + minute + m2 + C(half) + pre_level + pre_slope + margin'
specs={
 'Baseline':                base+' + brk',
 'Break$\\times$lead':       base+' + brk + brk_margin',
 'Break$\\times$lead$+$Elo':  base+' + brk + brk_margin + brk_elo',
 'Full':                    base+' + brk + brk_margin + brk_elo + brk_wbgt',
}
def fit(f): return smf.ols('Y ~ '+f,data=d).fit(cov_type='cluster',cov_kwds={'groups':d.match_id})
fits={k:fit(v) for k,v in specs.items()}
rowmap=[('Minute','minute'),('Minute$^2$','m2'),('Second half','C(half)[T.1]'),
 ('Pre-break level','pre_level'),('Pre-break slope','pre_slope'),('Score margin','margin'),
 ('Break','brk'),('Break $\\times$ lead','brk_margin'),
 ('Break $\\times$ Elo/100','brk_elo'),('Break $\\times$ heat/$^\\circ$C','brk_wbgt')]
def cell(f,key):
    if key not in f.params.index: return '--'
    return '$%+.2f$ (%.2f)'%(f.params[key],f.bse[key])
print(' | '.join(['Term']+list(specs.keys())))
for lab,key in rowmap:
    print(' | '.join([lab]+[cell(fits[k],key) for k in specs]))
with open('paper/coef_rows.tex','w') as fo:
    for lab,key in rowmap:
        fo.write(lab+' & '+' & '.join(cell(fits[k],key) for k in specs)+' \\\\\n')
    fo.write('\\midrule\n')
    fo.write('Match fixed effects & Yes & Yes & Yes & Yes \\\\\n')
    fo.write('Within $R^2$ & '+' & '.join('%.3f'%fits[k].rsquared for k in specs)+' \\\\\n')
print('\nwrote paper/coef_rows.tex ; break events=%d, control anchors=%d, matches=%d'%(int(d.brk.sum()),int((1-d.brk).sum()),d.match_id.nunique()))


Term | Baseline | Break$\times$lead | Break$\times$lead$+$Elo | Full
Minute | $-0.09$ (0.16) | $-0.09$ (0.16) | $-0.09$ (0.16) | $-0.09$ (0.16)
Minute$^2$ | $+0.00$ (0.00) | $+0.00$ (0.00) | $+0.00$ (0.00) | $+0.00$ (0.00)
Second half | $+2.41$ (4.21) | $+2.37$ (4.21) | $+2.39$ (4.20) | $+2.40$ (4.20)
Pre-break level | $+0.24$ (0.04) | $+0.24$ (0.04) | $+0.23$ (0.04) | $+0.23$ (0.04)
Pre-break slope | $+0.06$ (0.03) | $+0.06$ (0.03) | $+0.06$ (0.03) | $+0.06$ (0.03)
Score margin | $-2.31$ (1.20) | $-2.23$ (1.27) | $-2.24$ (1.26) | $-2.23$ (1.26)
Break | $+0.26$ (1.41) | $+0.38$ (1.49) | $+0.36$ (1.49) | $+0.35$ (1.49)
Break $\times$ lead | -- | $-0.90$ (1.39) | $-2.23$ (1.36) | $-2.19$ (1.37)
Break $\times$ Elo/100 | -- | -- | $+1.59$ (0.80) | $+1.58$ (0.81)
Break $\times$ heat/$^\circ$C | -- | -- | -- | $-0.21$ (0.31)

wrote paper/coef_rows.tex ; break events=198, control anchors=3139, matches=99


## 4 · Effect-vs-margin plot (Figure 5)

In [4]:
import numpy as np, statsmodels.formula.api as smf, warnings
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
exec(open('analysis_gap.py').read().split('if __name__')[0])
plt.rcParams.update({'font.family':'serif','font.size':11,'axes.spines.top':False,'axes.spines.right':False,
    'figure.dpi':140,'axes.grid':True,'grid.alpha':0.25})
INK='#1f2430'; BLUE='#2f6db5'; RED='#c0392b'; MUT='#6b7280'
d=build(10); rb='minute + m2 + C(half) + pre_level + pre_slope + margin'
X2=smf.mixedlm('Y ~ '+rb+' + brk + brk_margin',data=d,groups=d['match_id']).fit(method='lbfgs')
X3=smf.mixedlm('Y ~ '+rb+' + brk + brk_margin + brk_elo',data=d,groups=d['match_id']).fit(method='lbfgs')
def line(m, elo=None):
    b=m.params; V=m.cov_params(); mg=np.linspace(-3,3,121)
    val=b['brk']+b['brk_margin']*mg + (b['brk_elo']*elo if elo is not None else 0.0)
    var=V.loc['brk','brk']+mg**2*V.loc['brk_margin','brk_margin']+2*mg*V.loc['brk','brk_margin']
    return mg,val,1.96*np.sqrt(var)
fig,ax=plt.subplots(figsize=(8.0,4.7))
mg,v,e=line(X2)
ax.plot(mg,v,color=BLUE,lw=2,label='Break $\\times$ lead model (averaged over team strength)')
ax.fill_between(mg,v-e,v+e,color=BLUE,alpha=.15)
mg3,v3,e3=line(X3,elo=0.0)
ax.plot(mg3,v3,color=RED,lw=1.8,ls='--',label='Lead $+$ strength model, at average strength')
ax.axhline(0,color=INK,lw=1.0)
ax.set_xlabel('Oriented goal margin of the pre-break dominant side (negative $=$ behind, positive $=$ ahead)')
ax.set_ylabel('Implied break effect (sign-adjusted momentum points)')
ax.set_title('Break effect declines with the size of the lead')
ax.set_xticks(range(-3,4))
# data support: histogram of break-event margins along the bottom
t=d[d.brk==1]; import numpy as np
counts,edges=np.histogram(t.margin,bins=np.arange(-3.5,4.5,1))
y0=ax.get_ylim()[0]
for c,left in zip(counts,edges[:-1]):
    ax.bar(left+0.5,0.9*abs(y0)*c/counts.max(),bottom=y0,width=0.8,color=MUT,alpha=.20,zorder=0)
ax.legend(loc='upper right',fontsize=8.5,frameon=False)
ax.annotate('for scale: pre-break level $\\approx$ +20; natural fade $\\approx$ $-$12',
            xy=(0.015,0.04),xycoords='axes fraction',fontsize=8,color=MUT)
fig.tight_layout(); fig.savefig('paper/figures/fig4_effect_by_margin.png',bbox_inches='tight'); plt.close(fig)
print('continuous effect plot written')
for m in [-1,0,1,2]:
    b=X2.params; print('  X2 effect at margin %+d: %+.2f'%(m,b['brk']+b['brk_margin']*m))


continuous effect plot written
  X2 effect at margin -1: +1.28
  X2 effect at margin +0: +0.38
  X2 effect at margin +1: -0.52
  X2 effect at margin +2: -1.41


## 5 · Within-match play-aligned builder + ladder

In [5]:
"""
Design B (frozen-momentum / play-aligned) pipeline, mirroring analysis_gap.py.
Counterfactual: the break is dead time and momentum is frozen during it. The resume minute is play +1.
CASE/CONTROL matching (careful):
  - a CASE (break) at start minute s, with resume e = break end (per-match; group 'end', else s+3),
    has outcome window over PLAY minutes [e, e+H-1] (i.e. from resumption), pre-window [s-5,s-1], anchor = s.
  - a CONTROL (non-break) at minute c has outcome window over its immediate PLAY minutes [c+1, c+H]
    (continuous play, no break removed), pre-window [c-5,c-1], anchor = c.
  Both windows are H minutes of PLAY after the anchor; the covariates and time trend use the anchor minute,
  so the break contrast is play-aligned.  (Design A instead uses [c+GAP,c+GAP+H-1] for BOTH.)
"""
import json, numpy as np, pandas as pd, statsmodels.formula.api as smf, warnings
warnings.filterwarnings('ignore')
exec(open('analysis_gap.py').read().split('if __name__')[0])   # loaders, valid_len, DROP, GAP, elo_h/elo_a

END={}
for m in json.load(open('data/wc2026_group_momentum.json')):
    for b in m.get('breaks',[]):
        if 'start' in b and 'end' in b:
            w=b['end']-b['start']; END[(m['id'],b['start'])]=b['end'] if 2<=w<=4 else b['start']+3
def resume(mid,s): return END.get((mid,s), s+3)   # per-match resume minute (knockout imputed s+3)

def omean(mid,lo,hi):
    vs=[mom_at(mid,x) for x in range(lo,hi+1)]; vs=[v for v in vs if v is not None]
    return float(np.mean(vs)) if vs else None

def rowB(mid,c,brk,H):
    pre=wm(mid,c-5,c-1); sl=wsl(mid,c-5,c-1)
    if pre is None or sl is None: return None
    o=1 if pre>=0 else -1
    lo,hi=(resume(mid,c), resume(mid,c)+H-1) if brk else (c+1, c+H)   # play-aligned windows
    s,mo=raw[mid]; last=s+len(mo)-1
    if c-5<s or hi>last or period(c-5)!=period(hi): return None
    y=omean(mid,lo,hi)
    if y is None: return None
    hs,a=sb(mid,c); w=WB(mid); m=str(mid)
    if w is None or m not in elo_h: return None
    return dict(match_id=m,brk=brk,minute=c,m2=c*c,half=int(c>=45),wbgt=w,
                pre_level=o*pre,pre_slope=o*sl,margin=o*(hs-a),elo_gap=o*(elo_h[m]-elo_a[m]),Y=o*y)

def build_B(H):
    R=[]
    for mid,bks in breaks.items():
        if not valid_len(mid): continue
        for c in bks:
            r=rowB(mid,c,1,H)
            if r: R.append(r)
    for mid,(s,mo) in raw.items():
        if not valid_len(mid): continue
        for c in range(s+5, s+len(mo)-1):
            if any(b-2<=c<=b+GAP+H for b in breaks.get(mid,[])): continue
            if 43<=c<=48: continue
            r=rowB(mid,c,0,H)
            if r: R.append(r)
    d=pd.DataFrame(R); d=d[d.match_id!=DROP].copy()
    d['wbgt_c']=d.wbgt-d.wbgt.mean(); d['elo_c']=(d.elo_gap-d.elo_gap.mean())/100.0
    d['brk_margin']=d.brk*d.margin; d['brk_wbgt']=d.brk*d.wbgt_c; d['brk_elo']=d.brk*d.elo_c
    return d

if __name__=='__main__':
    d=build_B(10)
    nb=int(d.brk.sum()); nc=int((1-d.brk).sum()); nm=d.match_id.nunique()
    print(f"[DESIGN B] play-aligned, per-match resume | break n={nb}, control n={nc}, matches={nm}")
    def ols(f): return smf.ols("Y ~ "+f,data=d).fit(cov_type='cluster',cov_kwds={'groups':d.match_id})
    base="C(match_id) + minute + m2 + C(half) + pre_level + pre_slope + margin"
    def L(r,t):
        p=r.params[t]; s=r.bse[t]; return f"{p:+.2f} [{p-1.96*s:+.2f},{p+1.96*s:+.2f}] (p={r.pvalues[t]:.2f})"
    M0=ols(base+" + brk"); M1=ols(base+" + brk + brk_wbgt"); M2=ols(base+" + brk + brk_margin")
    M3=ols(base+" + brk + brk_elo"); M4=ols(base+" + brk + brk_margin + brk_elo + brk_wbgt")
    print("M0 base ", L(M0,'brk'))
    print("M2 +lead", L(M2,'brk'),"| xlead",L(M2,'brk_margin'))
    print("M4 full ", L(M4,'brk'),"| xlead",L(M4,'brk_margin'),"| xElo",L(M4,'brk_elo'),"| xWBGT",L(M4,'brk_wbgt'))
    def mixed(rhs): return smf.mixedlm("Y ~ "+rhs,data=d,groups=d["match_id"]).fit(method='lbfgs')
    rb="minute + m2 + C(half) + pre_level + pre_slope + margin"
    X1=mixed(rb+" + brk"); X2=mixed(rb+" + brk + brk_margin"); X3=mixed(rb+" + brk + brk_margin + brk_elo")
    print("X1 mixed", L(X1,'brk'))
    print("X3 mixed", L(X3,'brk'),"| xlead",L(X3,'brk_margin'),"| xElo",L(X3,'brk_elo'))


[DESIGN B] play-aligned, per-match resume | break n=198, control n=3157, matches=99


M0 base  -0.63 [-3.63,+2.37] (p=0.68)
M2 +lead -0.49 [-3.60,+2.63] (p=0.76) | xlead -1.04 [-3.70,+1.62] (p=0.45)
M4 full  -0.53 [-3.66,+2.59] (p=0.74) | xlead -2.37 [-4.96,+0.22] (p=0.07) | xElo +1.66 [+0.03,+3.28] (p=0.05) | xWBGT -0.30 [-0.92,+0.31] (p=0.33)
X1 mixed -0.63 [-3.31,+2.06] (p=0.65)
X3 mixed -0.51 [-3.21,+2.19] (p=0.71) | xlead -2.43 [-4.74,-0.12] (p=0.04) | xElo +1.66 [+0.47,+2.86] (p=0.01)


## 6 · Two-designs schematic (Figure 3)

In [6]:
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch
plt.rcParams.update({'font.family':'serif','font.size':10.5})
GOLD='#c98a1a'; GREY='0.6'; GREEN='#2e7d5b'; BLUE='#2f6db5'; INK='#1f2430'
fig,ax=plt.subplots(figsize=(9.2,4.3)); ax.set_xlim(-6,15); ax.set_ylim(0,3.4); ax.axis('off')
def band(x0,x1,y,color,alpha,label=None,hatch=None,ec='none'):
    ax.add_patch(Rectangle((x0,y-0.28),x1-x0,0.56,facecolor=color,alpha=alpha,hatch=hatch,edgecolor=ec,lw=0.8))
def lab(x,y,t,c=INK,fs=9,w='normal'): ax.text(x,y,t,ha='center',va='center',fontsize=fs,color=c,weight=w)
def rowlab(y,t): ax.text(-6.3,y,t,ha='right',va='center',fontsize=9.5,weight='bold')
# axis
ax.annotate('',xy=(15,0.25),xytext=(-6,0.25),arrowprops=dict(arrowstyle='->',color=INK,lw=1))
for m in range(-5,15,1): ax.plot([m,m],[0.2,0.3],color=INK,lw=.6)
for m in [-5,0,3,12]: ax.text(m,0.02,{-5:'$c-5$',0:'$c$',3:'$c+3$',12:'$c+12$'}[m],ha='center',fontsize=8,color=INK)
ax.text(14.5,0.02,'minutes',ha='right',fontsize=8,color=INK,style='italic')
# Row 1: break event (same under both designs)
y=2.8; rowlab(y,'Break event')
band(-5,-0.05,y,GOLD,.35); lab(-2.5,y,'pre-break',GOLD,8.5)
band(0,3,y,GREY,.45,hatch='//'); lab(1.5,y+0.02,'break',GREY,8)
band(3,12.05,y,GREEN,.35); lab(7.5,y,'post-resumption outcome',GREEN,8.5)
# Row 2: control under clock-aligned (Design A)
y=1.9; rowlab(y,'Control\n(Design A: clock)')
band(-5,-0.05,y,GOLD,.35); lab(-2.5,y,'pre',GOLD,8.5)
band(0,3,y,'0.85',.9,hatch='..',ec='0.5'); lab(1.5,y,'skipped',GREY,7.5)
band(3,12.05,y,BLUE,.30); lab(7.5,y,'outcome $[c{+}3,c{+}12]$',BLUE,8.5)
# Row 3: control under play-aligned (Design B)
y=1.0; rowlab(y,'Control\n(Design B: play)')
band(-5,-0.05,y,GOLD,.35); lab(-2.5,y,'pre',GOLD,8.5)
band(1,11.05,y,BLUE,.30); lab(6,y,'outcome $[c{+}1,c{+}10]$ (immediate)',BLUE,8.5)
# connecting note
ax.text(4.5,3.35,'The two designs differ only in the control window: clock-aligned skips three minutes to match the break;\nplay-aligned uses the immediate continuation, treating the break as removed (dead) time.',
        ha='center',va='top',fontsize=8.2,color=INK)
fig.tight_layout(); fig.savefig('paper/figures/fig_designs.png',bbox_inches='tight',dpi=150); plt.close(fig)
print('schematic done')


schematic done


## 7 · Combined Design A / Design B figure (Figure 4)

In [7]:
import json, numpy as np, warnings; warnings.filterwarnings('ignore')
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
exec(open('analysis_gap.py').read().split('if __name__')[0])   # loaders, valid_len, DROP, GAP
END={}
for m in json.load(open('data/wc2026_group_momentum.json')):
    for b in m.get('breaks',[]):
        if 'start' in b and 'end' in b:
            w=b['end']-b['start']; END[(m['id'],b['start'])]=b['end'] if 2<=w<=4 else b['start']+3
def resume(mid,s): return END.get((mid,s), s+3)
plt.rcParams.update({'font.family':'serif','font.size':10.5,'axes.spines.top':False,'axes.spines.right':False,
    'figure.dpi':140,'axes.grid':True,'grid.alpha':0.25,'legend.frameon':False})
RED='#c0392b'; BLUE='#2f6db5'; INK='#1f2430'
def om(mid,c,k,o): v=mom_at(mid,c+k); return o*v if v is not None else np.nan
def o_of(mid,c):
    p=wm(mid,c-5,c-1); return None if p is None else (1 if p>=0 else -1)
def okp(mid,c,lo,hi):
    s,mo=raw[mid]; last=s+len(mo)-1; return c+lo>=s and c+hi<=last and period(c+lo)==period(c+hi)
TR=[(m,c) for m,bks in breaks.items() if valid_len(m) and str(m)!=DROP for c in bks]
CT=[(m,c) for m,(s,mo) in raw.items() if valid_len(m) and str(m)!=DROP
    for c in range(s+5,s+len(mo)-16)
    if not any(b-2<=c<=b+GAP+15 for b in breaks.get(m,[])) and not (43<=c<=48)]
def mci(M): M=np.array(M); m=np.nanmean(M,0); se=np.nanstd(M,0)/np.sqrt(np.sum(np.isfinite(M),0)); return m,m-1.96*se,m+1.96*se

fig,ax=plt.subplots(1,2,figsize=(11,4.4),sharey=True)
# Panel A: clock-aligned (blank in-break 0,1,2)
K=np.arange(-5,16)
def clockcurve(anchors):
    R=[]
    for m,c in anchors:
        o=o_of(m,c)
        if o is None or not okp(m,c,-5,15): continue
        row=[]
        for k in K:
            if 0<=k<GAP: row.append(np.nan); continue
            row.append(om(m,c,k,o))
        R.append(row)
    return R
Bb=clockcurve(TR); Cc=clockcurve(CT)
mb,lb,hb=mci(Bb); mc,lc,hc=mci(Cc)
a=ax[0]; a.axvspan(-0.4,GAP-0.6,color='0.5',alpha=.28,hatch='//',ec='none'); a.axhline(0,color=INK,lw=.7)
a.plot(K,mb,color=RED,lw=1.7,marker='o',ms=3,label=f'break (n={len(Bb)})'); a.fill_between(K,lb,hb,color=RED,alpha=.13)
a.plot(K,mc,color=BLUE,lw=1.7,marker='s',ms=3,label=f'control (n={len(Cc)})'); a.fill_between(K,lc,hc,color=BLUE,alpha=.12)
a.set_title('(a) Design A: clock-aligned'); a.set_xlabel('Minutes relative to break start'); a.set_ylabel('Sign-adjusted momentum (dominant side)'); a.legend(loc='upper right',fontsize=8.5)
# Panel B: play-aligned
POST=np.arange(1,16); PRE=np.arange(-5,0)
def treatedB():
    R=[]
    for m,c in TR:
        o=o_of(m,c); e=resume(m,c)
        if o is None or not okp(m,c,-5,(e-c)+15): continue
        R.append([om(m,c,k,o) for k in PRE]+[om(m,e,k-1,o) for k in POST])
    return R
def controlB():
    R=[]
    for m,c in CT:
        o=o_of(m,c)
        if o is None or not okp(m,c,-5,15): continue
        R.append([om(m,c,k,o) for k in PRE]+[om(m,c,k,o) for k in POST])
    return R
Tb=treatedB(); Cb=controlB(); X=np.concatenate([PRE,POST])
mt,lt,ht=mci(Tb); mc2,lc2,hc2=mci(Cb)
b=ax[1]; b.axvspan(-0.5,0.5,color='0.6',alpha=.25,hatch='//',ec='none'); b.axhline(0,color=INK,lw=.7)
b.plot(X[:5],mt[:5],color=RED,lw=1.7,marker='o',ms=3); b.plot(X[5:],mt[5:],color=RED,lw=1.7,marker='o',ms=3,label=f'break (n={len(Tb)})')
b.fill_between(X[:5],lt[:5],ht[:5],color=RED,alpha=.13); b.fill_between(X[5:],lt[5:],ht[5:],color=RED,alpha=.13)
b.plot(X[:5],mc2[:5],color=BLUE,lw=1.7,marker='s',ms=3); b.plot(X[5:],mc2[5:],color=BLUE,lw=1.7,marker='s',ms=3,label=f'control (n={len(Cb)})')
b.fill_between(X[:5],lc2[:5],hc2[:5],color=BLUE,alpha=.12); b.fill_between(X[5:],lc2[5:],hc2[5:],color=BLUE,alpha=.12)
b.set_title('(b) Design B: play-aligned (break removed)'); b.set_xlabel('Play-time relative to break'); b.legend(loc='upper right',fontsize=8.5)
fig.tight_layout(); fig.savefig('paper/figures/fig_AB.png',bbox_inches='tight'); plt.close(fig)
print('fig_AB done; A post+3..+7 brk %.1f ctrl %.1f | B play+1..+3 brk %s ctrl %s'%(
   np.nanmean(mb[8:13]),np.nanmean(mc[8:13]),[round(x,1) for x in mt[5:8]],[round(x,1) for x in mc2[5:8]]))


fig_AB done; A post+3..+7 brk 6.5 ctrl 8.1 | B play+1..+3 brk [np.float64(9.2), np.float64(6.4), np.float64(4.2)] ctrl [np.float64(12.1), np.float64(10.3), np.float64(9.0)]


## 8 · External controls: dominance-oriented ATT, tight-caliper matching, CATEs

In [8]:
import json, numpy as np, pandas as pd, warnings; warnings.filterwarnings('ignore')
from scipy.optimize import minimize
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
rng=np.random.default_rng(11)
raw={}; goals={}
def add(fn):
    try: data=json.load(open(fn))
    except: return
    for m in data:
        if isinstance(m.get('mom'),list) and len(m['mom'])>10:
            raw[m['id']]=(m['startMin'],m['mom']); goals[m['id']]=m.get('goals',[])
for f in ['data/wc2026_knockout_raw.json','data/wc2026_group_momentum.json','data/wc_control_intl.json',
          'data/wc_control_pastwc.json','data/wc_control_group_momentum.json','data/wc_control_momentum.json']: add(f)
gv={x['id']:x for x in json.load(open('data/_gv.json'))}
_city={}
for _m in json.load(open('data/wc2026_knockout_raw.json')): _city[_m['id']]=_m.get('city')
for _mid,_g in gv.items(): _city.setdefault(_mid,_g.get('city'))
ACCITY={'Houston','Arlington','Dallas','Atlanta','Las Vegas','Glendale'}
def valid_len(mid):
    if mid not in raw: return False
    L=len(raw[mid][1]); return (80<=L<=100) or (118<=L<=128)
END={}
for m in json.load(open('data/wc2026_group_momentum.json')):
    for b in m.get('breaks',[]):
        if 'start' in b and 'end' in b:
            w=b['end']-b['start']; END[(m['id'],b['start'])]=b['end'] if 2<=w<=4 else b['start']+3
def resume(mid,s): return END.get((mid,s), s+3)
def wmean(mid,c,lo,hi):
    s,mom=raw[mid]; vs=[mom[c+k-s] for k in range(lo,hi+1) if 0<=c+k-s<len(mom)]; return np.mean(vs) if vs else None
def wslope(mid,c,lo,hi):
    s,mom=raw[mid]; p=[(k,mom[c+k-s]) for k in range(lo,hi+1) if 0<=c+k-s<len(mom)]
    if len(p)<2: return None
    x=np.array([a for a,_ in p]); y=np.array([b for _,b in p]); return np.polyfit(x,y,1)[0]
def opost(mid,c,o,lo,hi):
    s,mom=raw[mid]; vs=[mom[c+k-s] for k in range(lo,hi+1) if 0<=c+k-s<len(mom)]; return o*np.mean(vs) if vs else np.nan
def sb(mid,mn):
    hs=a=0
    for g in goals.get(mid,[]):
        if g['mn']+(g.get('at') or 0)/100.0<mn: hs+=g['h']; a+=(not g['h'])
    return hs,a
def ebal(Xc,tgt):
    Z=Xc-tgt
    def loss(l): a=-Z@l; a-=a.max(); return np.log(np.exp(a).sum())
    def grad(l): a=-Z@l; a-=a.max(); e=np.exp(a); w=e/e.sum(); return -(w@Z)
    r=minimize(loss,np.zeros(Z.shape[1]),jac=grad,method='L-BFGS-B',options={'maxiter':800})
    a=-Z@r.x; a-=a.max(); e=np.exp(a); return e/e.sum()

DROP='15186769'
trc=pd.DataFrame(json.load(open('data/treated_covariates.json')))
coc=pd.DataFrame(json.load(open('data/control_candidates_all.json')))
ELO_T={r['match_id']:(r['elo_home'],r['elo_away']) for _,r in trc.iterrows()}
def build_row(mid,c,treated,extra):
    if mid not in raw: return None
    pm=wmean(mid,c,-5,-1); sl=wslope(mid,c,-5,-1)
    if pm is None or sl is None: return None
    o=1 if pm>=0 else -1
    hs,a=sb(mid,c); smarg=o*(hs-a)
    yA=opost(mid,c,o,3,12)
    if treated:
        e=resume(mid,c); yB=opost(mid,c,o,e-c,e-c+9)
        eh,ea=ELO_T.get(mid,(np.nan,np.nan)); elo=o*(eh-ea)
        wb=20.7 if _city.get(mid) in ACCITY else extra['wbgt']
    else:
        yB=opost(mid,c,o,1,10)
        elo=o*extra['elo_gap_or']*extra['orient']       # de-orient then re-orient by dominance
        wb=extra['wbgt']
    if not np.isfinite(yA) or not np.isfinite(yB) or not np.isfinite(elo) or wb is None: return None
    return dict(match_id=mid,minute=c,treated=treated,half='H1' if c<45 else 'H2',
                wbgt=wb,local_hour=extra['local_hour'],elo=elo,smargin=smarg,
                pre_level=o*pm,pre_slope=o*sl,yA=yA,yB=yB)
rows=[]
for _,r in trc.iterrows():
    if not valid_len(r['match_id']) or r['match_id']==DROP: continue
    rr=build_row(r['match_id'],int(r['minute']),1,{'wbgt':r['wbgt'],'local_hour':r['local_hour']})
    if rr: rows.append(rr)
for _,r in coc.iterrows():
    rr=build_row(r['match_id'],int(r['minute']),0,{'wbgt':r['wbgt'],'local_hour':r['local_hour'],'elo_gap_or':r['elo_gap_or'],'orient':r['orient']})
    if rr: rows.append(rr)
F=pd.DataFrame(rows)
COV=['wbgt','local_hour','elo','smargin','minute','pre_level','pre_slope']
print('DOMINANCE-ORIENTED external analysis | treated=%d control=%d'%(int(F.treated.sum()),int((1-F.treated).sum())))
print('treated signed-margin dist:', dict(pd.Series(F[F.treated==1].smargin.astype(int)).value_counts().sort_index()))
mu=F[COV].mean().values; sd=F[COV].std().values
def att(col, sub=None):
    G=F if sub is None else F[sub]
    num=den=0
    for hf in ['H1','H2']:
        T=G[(G.half==hf)&(G.treated==1)]; C=G[(G.half==hf)&(G.treated==0)]
        if len(T)<3 or len(C)<10: continue
        Xt=(T[COV].values-mu)/sd; Xc=(C[COV].values-mu)/sd; w=ebal(Xc,Xt.mean(0))
        yt=np.nanmean(T[col].values); ya=C[col].values; fin=np.isfinite(ya)
        yc=np.sum(w[fin]*ya[fin])/np.sum(w[fin]); num+=len(T)*(yt-yc); den+=len(T)
    return num/den if den else np.nan
print('\n=== ENTROPY-BALANCED ATT (all on-support) ===')
print('  Design A (clock)  %+.2f'%att('yA'))
print('  Design B (play)   %+.2f'%att('yB'))
# tight caliper matching
Z=(F[COV].values-mu)/sd; tm=F.treated.values==1
ps=LogisticRegression(max_iter=1000).fit(Z,F.treated.values).predict_proba(Z)[:,1]
lg=np.log(np.clip(ps,1e-6,1-1e-6)/np.clip(1-ps,1e-6,1-1e-6)); cal=0.2*lg.std()
Ti=np.where(tm)[0]; Ci=np.where(~tm)[0]
d_,ix=NearestNeighbors(n_neighbors=1).fit(lg[Ci].reshape(-1,1)).kneighbors(lg[Ti].reshape(-1,1))
ok=d_[:,0]<=cal
print('\n=== TIGHT-CALIPER MATCHING (0.2 SD) | matched=%d dropped=%d ==='%(ok.sum(),(~ok).sum()))
drp=F.iloc[Ti[~ok]]
if len(drp): print('  dropped: WBGT %.1f, %%kickoff<=14h %.0f'%(drp.wbgt.mean(),100*(drp.local_hour<=14).mean()))
for lab,col in [('Design A','yA'),('Design B','yB')]:
    yt=F[col].values[Ti[ok]]; yc=F[col].values[Ci[ix[ok,0]]]; dd=(yt-yc); dd=dd[np.isfinite(dd)]
    se=dd.std()/np.sqrt(len(dd)); print('  %-9s matched ATT %+.2f [%+.2f,%+.2f]'%(lab,dd.mean(),dd.mean()-1.96*se,dd.mean()+1.96*se))
# CATE by lead-state (Design B)
def leadstate(m): return 'a_behind' if m<0 else ('b_level' if m==0 else ('c_ahead1' if m==1 else 'd_ahead2+'))
F['lead']=[leadstate(m) for m in F.smargin]
def cate_ci(sub, cov, col='yB'):
    G=F[sub]; T=G[G.treated==1]; C=G[G.treated==0]
    if len(T)<5 or len(C)<20: return None,len(T),(np.nan,np.nan)
    m0=F[cov].mean().values; s0=F[cov].std().values
    def one(T,C):
        Xt=(T[cov].values-m0)/s0; Xc=(C[cov].values-m0)/s0; w=ebal(Xc,Xt.mean(0))
        ya=C[col].values; fin=np.isfinite(ya); return np.nanmean(T[col].values)-np.sum(w[fin]*ya[fin])/np.sum(w[fin])
    est=one(T,C); ut=T.match_id.unique(); uc=C.match_id.unique(); bs=[]
    for _ in range(200):
        tb=T[T.match_id.isin(rng.choice(ut,len(ut),True))]; cb=C[C.match_id.isin(rng.choice(uc,len(uc),True))]
        if len(tb)<5 or len(cb)<20: continue
        try: bs.append(one(tb,cb))
        except: pass
    lo,hi=np.percentile(bs,[2.5,97.5]) if bs else (np.nan,np.nan); return est,len(T),(lo,hi)
COV2=[c for c in COV if c!='smargin']
print('\n=== CATE by dominance-signed lead (Design B, external) ===')
for L in ['a_behind','b_level','c_ahead1','d_ahead2+']:
    a,n,ci=cate_ci(F.lead==L,COV2)
    print('  %-10s n=%3d  ATT_B %s'%(L,n,'%+.2f [%+.2f,%+.2f]'%(a,ci[0],ci[1]) if a is not None else '(thin)'))
print('\n=== CATE by heat (Design B, external) ===')
for L,mask in [('cool WBGT<28',F.wbgt<28),('hot WBGT>=28',F.wbgt>=28)]:
    a,n,ci=cate_ci(mask,COV)
    print('  %-14s n=%3d  ATT_B %s'%(L,n,'%+.2f [%+.2f,%+.2f]'%(a,ci[0],ci[1]) if a is not None else '(thin)'))


DOMINANCE-ORIENTED external analysis | treated=199 control=8946
treated signed-margin dist: {-4: np.int64(1), -3: np.int64(3), -2: np.int64(11), -1: np.int64(26), 0: np.int64(98), 1: np.int64(41), 2: np.int64(11), 3: np.int64(5), 4: np.int64(3)}

=== ENTROPY-BALANCED ATT (all on-support) ===
  Design A (clock)  +0.56
  Design B (play)   -0.59

=== TIGHT-CALIPER MATCHING (0.2 SD) | matched=196 dropped=3 ===
  dropped: WBGT 23.8, %kickoff<=14h 100
  Design A  matched ATT -1.57 [-5.00,+1.87]
  Design B  matched ATT -2.48 [-5.97,+1.01]

=== CATE by dominance-signed lead (Design B, external) ===


  a_behind   n= 41  ATT_B +1.39 [-3.78,+6.48]


  b_level    n= 98  ATT_B -0.11 [-2.59,+2.57]


  c_ahead1   n= 41  ATT_B +4.58 [-2.04,+10.30]


  d_ahead2+  n= 19  ATT_B -2.21 [-9.46,+20.76]

=== CATE by heat (Design B, external) ===


  cool WBGT<28   n=156  ATT_B +0.43 [-2.38,+3.03]


  hot WBGT>=28   n= 43  ATT_B -4.60 [-9.58,+7.61]


## 9 · Event-based check: net xG under both designs (Section 6.1)
xG has a true zero during the dead break window, so it is free of the momentum index's rebuild-lag artefact. Computed on the same 99-match / 198-event sample and the same clock- and play-aligned windows as the momentum designs, the break effect on net xG is essentially zero under both, confirming the transient play-aligned momentum dip is a property of the index rather than of performance.

In [9]:
import json, numpy as np, pandas as pd, statsmodels.formula.api as smf, warnings
warnings.filterwarnings('ignore')
exec(open('analysis_gap.py').read().split('if __name__')[0])   # raw,breaks,WB,wm,wsl,sb,period,valid_len,DROP,GAP
XG={int(k):v for k,v in json.load(open('data/xg_2026.json')).items()}

# per-match resume minute (as in the Design B / play-aligned builder)
END={}
for m in json.load(open('data/wc2026_group_momentum.json')):
    for b in m.get('breaks',[]):
        if 'start' in b and 'end' in b:
            w=b['end']-b['start']; END[(m['id'],b['start'])]=b['end'] if 2<=w<=4 else b['start']+3
def resume(mid,s): return END.get((str(mid),s), END.get((mid,s), s+3))

def netxg_window(mid,o,lo,hi):
    if mid not in XG: return None
    s=0.0
    for sh in XG[mid]:
        if lo <= sh['t'] <= hi: s+= sh['xg']*(1 if sh['h'] else -1)
    return o*s

def build_xg(design):
    R=[]
    def add(mid,c,brk):
        pre=wm(mid,c-5,c-1); sl=wsl(mid,c-5,c-1)
        if pre is None or sl is None: return
        s,mo=raw[mid]; last=s+len(mo)-1
        o=1 if pre>=0 else -1
        if design=='A':                       # clock-aligned: [c+3, c+12] for both
            lo,hi=c+GAP, c+GAP+9
        else:                                 # play-aligned: break from resume, control immediate
            lo,hi=(resume(mid,c), resume(mid,c)+9) if brk else (c+1, c+10)
        if c-5<s or hi>last or period(c-5)!=period(hi): return
        y=netxg_window(mid,o,lo,hi)
        if y is None: return
        hs,a=sb(mid,c); w=WB(mid)
        if w is None: return
        R.append(dict(match_id=str(mid),brk=brk,minute=c,m2=c*c,half=int(c>=45),
                      pre_level=o*pre,pre_slope=o*sl,margin=o*(hs-a),Y=y))
    for mid,bks in breaks.items():
        if not valid_len(mid): continue
        for c in bks: add(mid,c,1)
    for mid,(s,mo) in raw.items():
        if not valid_len(mid): continue
        for c in range(s+5, s+len(mo)-1):
            if any(b-2<=c<=b+GAP+10 for b in breaks.get(mid,[])): continue
            if 43<=c<=48: continue
            add(mid,c,0)
    d=pd.DataFrame(R); d=d[d.match_id!=DROP].copy()
    return d[d.match_id.astype(int).isin(XG.keys())].copy()

print("Net-xG break effect (dominance-oriented, 10-min post window), cluster-robust on match:")
for design,lab in [('A','clock-aligned'),('B','play-aligned ')]:
    d=build_xg(design)
    m=smf.ols("Y ~ C(match_id) + minute + m2 + C(half) + pre_level + pre_slope + margin + brk",
              data=d).fit(cov_type='cluster',cov_kwds={'groups':d.match_id})
    b=m.params['brk']; se=m.bse['brk']
    print("  [%s] %+.3f  95%% CI [%+.3f, %+.3f]  (p=%.2f)  n_break=%d n_ctrl=%d matches=%d"
          %(lab,b,b-1.96*se,b+1.96*se,m.pvalues['brk'],int(d.brk.sum()),int((1-d.brk).sum()),d.match_id.nunique()))

Net-xG break effect (dominance-oriented, 10-min post window), cluster-robust on match:
  [clock-aligned] -0.001  95% CI [-0.051, +0.049]  (p=0.98)  n_break=198 n_ctrl=3139 matches=99


  [play-aligned ] +0.011  95% CI [-0.051, +0.073]  (p=0.73)  n_break=198 n_ctrl=3157 matches=99
